# 강화학습 3강 — 기억하기 (리플레이 메모리)

2강에서는 한 걸음마다 경험을 **한 번 배우고 버렸습니다.**
사람은 그렇지 않죠. 지난 경험을 기억해 두었다가 다시 떠올리며 배웁니다.

오늘은 경험을 **기억 상자(리플레이 메모리)** 에 저장하고, 무작위로 꺼내 다시 배웁니다.
그리고 탐험을 점점 줄여서 **이익이 되는 행동만** 남깁니다.

> 위에서부터 순서대로 ▶ 실행하세요.

## 1. 2강에서 만든 것 (칸 · 점수표 · 행동 고르기 · 점수표 고치기)

In [ ]:
import gymnasium as gym
import numpy as np
import random

# CartPole 의 상태는 숫자 4개입니다: [수레 위치, 수레 속도, 막대 각도, 막대 회전 속도]
# 숫자가 너무 잘게 나뉘어 있어서, 비슷한 상태끼리 「칸」으로 묶어 줍니다.
위치_경계 = [-0.8, 0.8]                        # 왼쪽 / 가운데 / 오른쪽  → 3칸
속도_경계 = [-0.5, 0.5]                        # 왼쪽으로 / 거의 멈춤 / 오른쪽으로 → 3칸
각도_경계 = np.linspace(-0.21, 0.21, 7)[1:-1]   # 막대가 얼마나 기울었나 → 6칸
회전_경계 = np.linspace(-2.0, 2.0, 7)[1:-1]     # 막대가 얼마나 빨리 넘어가나 → 6칸

def 칸_찾기(state):
    """상태(숫자 4개) → 칸 번호 4개"""
    return (
        int(np.digitize(state[0], 위치_경계)),
        int(np.digitize(state[1], 속도_경계)),
        int(np.digitize(state[2], 각도_경계)),
        int(np.digitize(state[3], 회전_경계)),
    )

# 점수표(Q표) 만들기 — 칸마다 [왼쪽으로 밀기 점수, 오른쪽으로 밀기 점수]
def 새_점수표():
    return np.zeros((3, 3, 6, 6, 2))

# 행동 고르기 — 엡실론(ε) 확률로 「탐험」, 나머지는 「활용」
def 행동_고르기(점수표, 칸, 엡실론):
    if random.random() < 엡실론:
        return random.randint(0, 1)          # 탐험: 아무거나 해 본다 (1강의 무작위 선택)
    return int(np.argmax(점수표[칸]))         # 활용: 지금까지 점수가 가장 높은 행동

# 점수표 고치기 — 「방금 받은 보상 + 다음 칸에서 기대되는 점수」 쪽으로 조금씩 옮긴다
배우는_속도 = 0.1    # 한 번에 얼마나 고칠까
미래_할인 = 0.99     # 미래 점수를 얼마나 믿을까

def 점수표_고치기(점수표, 칸, 행동, 보상, 다음_칸, 끝남):
    목표 = 보상 if 끝남 else 보상 + 미래_할인 * np.max(점수표[다음_칸])
    점수표[칸 + (행동,)] += 배우는_속도 * (목표 - 점수표[칸 + (행동,)])

print("준비 완료")

## 2. 기억 상자 만들기

경험 하나 = (지금 칸, 한 행동, 받은 보상, 다음 칸, 끝났는지)
상자가 가득 차면 가장 오래된 기억부터 지워집니다.

In [ ]:
from collections import deque

기억_상자 = deque(maxlen=10000)     # 최대 1만 개까지 기억

env = gym.make("CartPole-v1")
state, info = env.reset()
for _ in range(5):
    행동 = random.randint(0, 1)
    next_state, reward, terminated, truncated, info = env.step(행동)
    기억_상자.append((칸_찾기(state), 행동, reward, 칸_찾기(next_state), terminated))   # 저장
    state = next_state
env.close()

print("기억 개수:", len(기억_상자))
print("무작위로 2개 꺼내 보기:")
for 기억 in random.sample(기억_상자, 2):
    print("  ", 기억)

## 3. 기억하면서 배우기

In [ ]:
def 학습하기(기억_쓰기, 판수=300, 한번에_꺼낼_기억=32):
    env = gym.make("CartPole-v1")
    점수표 = 새_점수표()
    기억_상자 = deque(maxlen=10000)
    엡실론 = 1.0
    점수들 = []

    for 판 in range(1, 판수 + 1):
        state, info = env.reset()
        칸 = 칸_찾기(state)
        끝 = False
        점수 = 0
        while not 끝:
            행동 = 행동_고르기(점수표, 칸, 엡실론)
            next_state, reward, terminated, truncated, info = env.step(행동)
            끝 = terminated or truncated
            다음_칸 = 칸_찾기(next_state)

            if 기억_쓰기:
                기억_상자.append((칸, 행동, reward, 다음_칸, terminated))       # ① 저장
                if len(기억_상자) >= 64:
                    for 기억 in random.sample(기억_상자, 한번에_꺼낼_기억):     # ② 무작위로 꺼내
                        점수표_고치기(점수표, *기억)                             # ③ 다시 배운다
            else:
                점수표_고치기(점수표, 칸, 행동, reward, 다음_칸, terminated)   # 한 번 배우고 버림

            칸 = 다음_칸
            점수 += reward
        점수들.append(점수)
        엡실론 = max(0.01, 엡실론 * 0.98)      # 탐험을 점점 줄이고 → 이익이 되는 행동만 남긴다

        if 판 % 50 == 0:
            이름 = "기억 상자" if 기억_쓰기 else "기억 없이"
            print(f"[{이름}] {판}판 · 탐험 비율 {엡실론:.2f} · 최근 50판 평균 {np.mean(점수들[-50:]):.0f}점")
    env.close()
    return 점수들, 점수표

## 4. 기억 없이 vs 기억 상자 (같은 300판)

In [ ]:
기억없이_점수, _ = 학습하기(기억_쓰기=False)
print()
기억상자_점수, 내_점수표 = 학습하기(기억_쓰기=True)

print("\n===== 100판씩 평균 =====")
print("            1~100판  101~200판  201~300판")
for 이름, 점수들 in [("기억 없이", 기억없이_점수), ("기억 상자", 기억상자_점수)]:
    구간 = [np.mean(점수들[i:i + 100]) for i in (0, 100, 200)]
    print(f"{이름} : {구간[0]:7.0f}  {구간[1]:9.0f}  {구간[2]:9.0f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
for 이름, 점수들 in [("no memory", 기억없이_점수), ("replay memory", 기억상자_점수)]:
    plt.plot(np.convolve(점수들, np.ones(20) / 20, mode="valid"), label=이름)
plt.xlabel("episode")
plt.ylabel("score (20-episode average)")
plt.legend()
plt.show()

## 5. 배운 두뇌 저장하고, 다시 불러와 플레이

배운 점수표를 파일로 저장합니다. 왼쪽 📁 파일 창에 `cartpole_brain.npy` 가 생깁니다.

In [ ]:
np.save("cartpole_brain.npy", 내_점수표)
print("저장 완료: cartpole_brain.npy")

불러온_점수표 = np.load("cartpole_brain.npy")
env = gym.make("CartPole-v1")
for 판 in range(1, 6):
    state, info = env.reset()
    끝 = False
    점수 = 0
    while not 끝:
        행동 = 행동_고르기(불러온_점수표, 칸_찾기(state), 엡실론=0)   # 이제는 이익이 되는 행동만
        state, reward, terminated, truncated, info = env.step(행동)
        끝 = terminated or truncated
        점수 += reward
    print(f"{판}판: {점수:.0f}점")
env.close()

## 정리

1. **저장** — 겪은 일을 기억 상자에 넣는다
2. **꺼내기** — 순서대로가 아니라 무작위로 꺼낸다 (방금 일만 계속 배우지 않도록)
3. **다시 배우기** — 같은 경험으로 여러 번 배우니, 같은 판수로도 더 빨리 잘하게 된다
4. **탐험 줄이기** — ε 를 점점 줄여서 마지막엔 이익이 되는 행동만 남긴다

**다음 4강** — 지금 점수표는 칸이 324개뿐입니다. 벽돌깨기처럼 화면 전체가 상태라면 칸이 셀 수 없이 많아집니다. 그래서 점수표 대신 **딥러닝**이 점수를 계산하게 합니다. 그게 DQN 입니다.